# Módulo 4 · Sesión 6 · Cuaderno de trabajo
**Martes 27 de octubre de 2026.** Cubre 4.1 (agentes), 4.2 (procesar texto, **10 puntos**), 4.3 y 4.4 (confiar en un modelo).

**Hoy no se entrega nada.** Todo se hace aquí, en clase. Subir este cuaderno después es opcional y da puntaje extra (el tope se confirma mañana).
**Los datos y los textos de este cuaderno son inventados para la clase.** No son del BCE ni del INEC.
**Construye el proyecto final:** la §7 (criterio entre regla y modelo).

**Cómo se usa:** corra las celdas en orden. Las marcadas **[COMPLETE]** tienen `___`: reemplácelos. **Descargue el `.ipynb` antes de cerrar Colab.**
**Plan B:** si algo falla, avise en el chat; puede seguir leyendo la salida en la pantalla compartida.

## 0 · Preparación

In [ ]:
# Celda 0 - prepara la carpeta de trabajo. En Colab trae el repositorio; en Codespaces ya esta.
import os, sys, subprocess
if not os.path.exists("regex_demo.py"):
    if not os.path.exists("m4-bd-ia-bce"):
        subprocess.run(["git", "clone", "https://github.com/jonathantitoec/m4-bd-ia-bce"], check=True)
    os.chdir("m4-bd-ia-bce")
print("carpeta:", os.getcwd())
print("archivos del kit:", [f for f in ("regex_demo.py","clasificador_demo.py","agente_vigilante.py","avisar.py") if os.path.exists(f)])

## 4.1 · Un agente con límites
**Un agente observa, decide y actúa, en ciclo, hasta cumplir su objetivo o detenerse.** `agente_vigilante.py` vigila `datos/exportaciones.csv`: si cambia, corre el flujo. Tiene **cinco vueltas como máximo**, **dos fallos seguidos y para**, y su aviso sale como **BORRADOR** hasta que una persona lo apruebe. Cada paso queda en `bitacora.jsonl`.

In [ ]:
# Celda 4.1a - corre el agente y, a mitad de la corrida, toca el archivo para que "cambie".
import time, pathlib
if os.path.exists("bitacora.jsonl"): os.remove("bitacora.jsonl")
agente = subprocess.Popen([sys.executable, "agente_vigilante.py"], stdout=subprocess.PIPE, text=True)
time.sleep(5)
pathlib.Path("datos/exportaciones.csv").touch()      # la fuente "cambia"
print("resultado:", agente.communicate()[0].strip())

In [ ]:
# Celda 4.1b - la bitacora: que observo, que decidio, que hizo.
import json
for linea in open("bitacora.jsonl", encoding="utf-8"):
    r = json.loads(linea); print(r["paso"], "|", r["decido"], "|", r["hice"])

**[COMPLETE] Pregunta 4.1 (una línea, para el chat):** en un proceso de su área, ¿qué herramienta le daría a un agente y cuál **no** le daría nunca? Escríbala aquí:

In [ ]:
RESPUESTA_4_1 = "___"      # [COMPLETE]
print(RESPUESTA_4_1)

## 4.2 · Procesar texto automáticamente *(10 puntos · individual · 6 minutos)*
Un boletín es **texto corrido**, sin columnas. Dos herramientas: una **regla** (expresión regular) y un **modelo** (clasificador). Primero cargamos las dos, ya escritas (`regex_demo.py` y `clasificador_demo.py`). Sus salidas se ocultan: no hace falta leerlas.

In [ ]:
%%capture
# Celda 4.2a - carga la regla (patron, convertir, p_fecha, p_pct, MESES) y el modelo (preprocess, vec, mod).
%run regex_demo.py
%run clasificador_demo.py

**Los dos ensayos.** *Ensayar* una regla es correrla donde usted ya conoce la respuesta. Las dos que siguen **fallan sin avisar**: obsérvelas.

In [ ]:
# Celda 4.2b - ensayo 1: el patron sobre una cifra SIN puntos de miles. Esperado: 3253.37
print("ensayo 1:", patron.findall("Las exportaciones sumaron 3253,37 millones de USD."))
# ensayo 2: invertir el orden de los replace en convertir. Esperado: 3253.37
convertir_mal = lambda s: float(s.replace(",", ".").replace(".", ""))
print("ensayo 2:", convertir_mal("3.253,37"))
# y float directo ni siquiera lo intenta:
try: float("3.253,37")
except ValueError as e: print("float directo:", e)

**El boletín 2** (inventado). Las tres celdas que siguen son **lo que se califica**. Pegue sus salidas en el chat **antes de las 20:08**.

In [ ]:
boletin2 = ("Las exportaciones petroleras sumaron 1.104,60 millones de USD, con corte al 15 de octubre de 2026. "
            "Las no petroleras llegaron a 2.049,77 millones de USD, con un aumento de 3,2% frente al mes anterior. "
            "Las remesas fueron 412,05 millones de USD.")

In [ ]:
# [COMPLETE] Celda 4.2-1 - tres cifras, convertidas a NUMERO (no texto).
cifras = [convertir(t) for t in patron.findall(___)]       # ___ = el texto donde buscar
print("CIFRAS:", cifras)

In [ ]:
# [COMPLETE] Celda 4.2-2 - fecha del corte (AAAA-MM-DD) y porcentaje.
d, mes, a = p_fecha.search(___).groups()
fecha = f"{a}-{MESES[mes]:02d}-{int(d):02d}"
pct = float(p_pct.search(___).group(1).replace(",", "."))
print("FECHA:", fecha, "| PORCENTAJE:", pct)

In [ ]:
# [COMPLETE] Celda 4.2-3 - tres frases SUYAS: tema, confianza y decision (aceptar o revisar, con un porque).
UMBRAL = 0.6          # confianza minima para aceptar sin revision
mis_frases = [
    "___",            # una frase de su area, que el modelo deberia reconocer
    "___",            # una frase de su area, pero con palabras que el modelo no vio
    "___",            # una frase que NO sea del dominio
]
porques = ["___", "___", "___"]       # una linea por frase: por que acepta o revisa
for frase, porque in zip(mis_frases, porques):
    p = mod.predict_proba(vec.transform([preprocess(frase)]))[0]
    conf = p.max()
    decision = "aceptar" if conf >= UMBRAL else "revisar"
    print("FRASE:", frase[:50], "| TEMA:", mod.classes_[p.argmax()], "| CONFIANZA:", round(conf, 2), "| DECISION:", decision, "|", porque)

**Pegue en el chat las líneas `CIFRAS`, `FECHA`/`PORCENTAJE` y las tres líneas `FRASE`.**

## 4.3 · ¿Regla o modelo?
Tres preguntas: **1)** ¿Puedo escribir la regla completa? → regla. **2)** ¿Hay una cifra de control exacta? → regla. **3)** ¿Los casos cambian de forma que ninguna regla prevé, y hay ejemplos y quien revise? → modelo, con validación.
**Un grupo puede concluir que su proceso no necesita modelo, y eso cuenta como cumplido.** Declarar por qué es el trabajo; se hace en la §7 del proyecto.

## 4.4 · Confiar en un modelo
Tres comprobaciones: **1)** cuánto acierta con casos que **no vio**; **2)** si le gana a **responder siempre lo más frecuente** (línea base); **3)** qué tan segura está **cada respuesta** (la mayor probabilidad).

In [ ]:
# Celda 4.4a - comprobaciones 1 y 2: exactitud de prueba contra la linea base.
print("linea base:", round(base.score(B, yte), 2), "| modelo:", round(mod.score(B, yte), 2))

**Pregunta al chat:** si el modelo acierta lo mismo que la línea base, ¿vale la pena usarlo?

In [ ]:
# Celda 4.4b - comprobacion 3: la confianza de cuatro frases y cuantas pasan a revision segun el umbral.
FRASES = ["Las exportaciones de camarón sumaron 3.253,37 millones de USD.",
          "El desempleo bajó en el trimestre.",
          "El club ganó el partido de ayer por dos goles.",
          "Se aprobó el presupuesto de la institución para el próximo año."]
conf = [mod.predict_proba(vec.transform([preprocess(t)]))[0].max() for t in FRASES]
for t, c in zip(FRASES, conf): print(round(c, 2), "|", t)
for u in (0.4, 0.5, 0.9):
    print("umbral", u, "-> a revision:", sum(c < u for c in conf), "de", len(FRASES))

**[COMPLETE] Decisión de su área.** Con un umbral alto van más frases a revisión. ¿Quién las revisaría en su área y cuántas puede revisar al día? Un umbral se decide **mirando los errores**, no se supone: estas frases son inventadas.

In [ ]:
UMBRAL_PROPUESTO = ___          # [COMPLETE] un numero entre 0 y 1
QUIEN_REVISA = "___"            # [COMPLETE]
REVISIONES_POR_DIA = ___        # [COMPLETE]
print(UMBRAL_PROPUESTO, QUIEN_REVISA, REVISIONES_POR_DIA)

**Cierre.** Descargue el `.ipynb` (*Archivo → Descargar → .ipynb*) **antes de cerrar**. Mañana presentan el cuaderno del proyecto, corriendo.